# Task 3 — Four-model saved-record audit
CPU-only audit of four saved 69-case static development diagnostics (276 attempts). The audit verifies manifests, development membership, original status labels, chosen-token evidence, usage totals and session accounting with the pinned generation-side validator.
Whole-output uncertainty values are exported only for complete responses with valid evidence. Truncated and invalid-evidence records retain unavailable scores and all incurred token/time costs. Parser counts are saved metadata; official correctness and meaningful-token eligibility are evaluated separately.
Original model records remain unchanged. Output: `MyDrive/UQRoute-TC/throughput/four-model-static-strata-audit-v1/record_audit.json`. This is a development diagnostic, not held-out evaluation or a routing-effectiveness analysis.
Runtime: Colab CPU. Execute setup and audit cells in order.


In [ ]:
from pathlib import Path
import ast, collections, hashlib, json, math, subprocess
from typing import Any
from google.colab import drive
drive.mount('/content/drive')
CODE_REVISION = 'adb48d1cfea33f4c10394e75a85e2cbdbaf6fe53'
project = Path('/content/uqroute-tc-record-audit')
if not project.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/RavindraSSK/uqroute-tc.git',str(project)], check=True)
    subprocess.run(['git','-C',str(project),'checkout','--detach',CODE_REVISION], check=True)
assert subprocess.check_output(['git','-C',str(project),'rev-parse','HEAD'], text=True).strip() == CODE_REVISION
assert not subprocess.check_output(['git','-C',str(project),'status','--porcelain'], text=True).strip()
sha = lambda b: hashlib.sha256(b).hexdigest()
pilot_path = project / 'src/uqroute_tc/inference/pilot.py'
assert sha(pilot_path.read_bytes()) == 'b7c1c56c035dab08d077e42021c45052e81f9b4558b4224ca36914d055f84aad'
function = next(n for n in ast.parse(pilot_path.read_text()).body if isinstance(n,ast.FunctionDef) and n.name=='token_evidence')
namespace = {'Any':Any,'math':math}
exec(compile(ast.Module(body=[function],type_ignores=[]),'pinned_token_evidence','exec'),namespace)
token_evidence = namespace['token_evidence']
freeze_path = project / 'docs/config/task1_single_sample_v1.json'
freeze = json.loads(freeze_path.read_text())
split_path = project / freeze['benchmark']['split_manifest']
split = json.loads(split_path.read_text())
root = Path('/content/drive/MyDrive/UQRoute-TC/throughput')
model_ids = {'qwen15b':'Qwen/Qwen2.5-1.5B-Instruct','llama3b':'meta-llama/Llama-3.2-3B-Instruct','qwen7b':'Qwen/Qwen2.5-7B-Instruct','qwen14b_awq':'Qwen/Qwen2.5-14B-Instruct-AWQ'}
print('Pinned validator loaded; four saved model folders selected.')


In [ ]:
models = {}
shared_keys = None
expected_counts = {'qwen15b':{'complete':69},'llama3b':{'complete':61,'truncated':8},'qwen7b':{'complete':68,'truncated':1},'qwen14b_awq':{'complete':62,'invalid_evidence':5,'truncated':2}}
for model_key, model_id in model_ids.items():
    folder = root / (model_key + '-static-strata-v1')
    manifest_path, summary_path = folder/'manifest.json', folder/'summary.json'
    manifest, summary = json.loads(manifest_path.read_text()), json.loads(summary_path.read_text())
    assert manifest['model_id'] == model_id
    assert manifest['model_settings'] == next(m for m in freeze['models'] if m['model_id']==model_id)
    assert manifest['freeze_sha256'] == sha(freeze_path.read_bytes())
    assert manifest['split_sha256'] == sha(split_path.read_bytes())
    assert manifest['pilot_sha256'] == sha(pilot_path.read_bytes())
    assert manifest['code_revision'] == CODE_REVISION
    assert manifest['benchmark_revision'] == freeze['benchmark']['revision']
    assert manifest['server_settings'] == freeze['single_sample']['server']
    for key in ['temperature','max_tokens','top_logprobs','request_seed']:
        assert manifest[key] == freeze['single_sample'][key]
    assert manifest['concurrency'] == 1 and manifest['automatic_retries'] == 0
    expected = set(manifest['case_keys'])
    assert len(expected) == len(manifest['case_keys']) == 69
    if shared_keys is None:
        shared_keys = expected
        shared_source_hashes = manifest['source_hashes']
    assert expected == shared_keys and manifest['source_hashes'] == shared_source_hashes
    paths = sorted((folder/'cases').glob('*/attempts/*.json'))
    records = [json.loads(p.read_text()) for p in paths]
    assert len(records) == 69 and {r['case_key'] for r in records} == expected
    assert len({r['case_key'] for r in records}) == len(records)
    counts = dict(collections.Counter(r['status'] for r in records))
    assert counts == summary['status_counts'] == expected_counts[model_key]
    assert summary['attempt_count'] == 69
    assert math.isclose(sum(r['elapsed_seconds'] for r in records),summary['measured_attempt_seconds'],rel_tol=1e-9)
    for key in ['prompt_tokens','completion_tokens','total_tokens']:
        known=[r['usage'][key] for r in records if isinstance(r.get('usage'),dict) and r['usage'].get(key) is not None]
        assert sum(known) == summary['known_token_totals'][key]
        assert 69-len(known) == summary['missing_usage_attempts'][key]
    rows=[]
    for path, record in zip(paths,records):
        assert path.parent.parent.name == sha(record['case_key'].encode())
        group = split['assignments'][record['base_id']]
        assert group['population']=='primary' and group['split']=='development'
        assert record['case_key']==record['source_file']+':'+record['id']
        response=record['response']; assert len(response['choices'])==1
        choice=response['choices'][0]
        assert record['usage']==response['usage'] and record['finish_reason']==choice['finish_reason']
        assert record['prediction']['raw_output']==(choice['message'].get('content') or '')
        replay=token_evidence(choice)
        assert replay==record['evidence'], record['case_key']
        derived_status='truncated' if choice['finish_reason']=='length' else 'complete' if replay['valid'] else 'invalid_evidence'
        assert record['status']==derived_status
        eligible=derived_status=='complete' and replay['valid']
        scores={k:replay[k] for k in ['sequence_nll','mean_token_nll','max_token_surprisal']} if eligible else None
        rows.append({k:record[k] for k in ['case_key','base_id','benchmark','source_file','status','finish_reason','session_id','usage','elapsed_seconds']} | {'record_sha256':sha(path.read_bytes()),'token_evidence_valid':replay['valid'],'evidence_reason':replay['reason'],'whole_output_score_eligible':eligible,'whole_output_scores':scores,'saved_parser_call_count':len(record['prediction']['tool_calls'])})
    sessions=[json.loads(p.read_text()) for p in sorted((folder/'sessions').glob('*.json'))]
    assert len(sessions)==summary['session_count']
    assert all(s['status']=='finished' for s in sessions)
    assert summary['unfinished_sessions']==0
    assert math.isclose(sum(s['batch_wall_seconds'] for s in sessions),summary['measured_batch_wall_seconds'],rel_tol=1e-9)
    assert {r['session_id'] for r in records} <= {s['id'] for s in sessions}
    models[model_key]={'model_id':model_id,'manifest_sha256':sha(manifest_path.read_bytes()),'summary_sha256':sha(summary_path.read_bytes()),'status_counts':counts,'token_evidence_valid_count':sum(r['token_evidence_valid'] for r in rows),'whole_output_score_eligible_count':sum(r['whole_output_score_eligible'] for r in rows),'aggregate_accounting':summary,'sessions':sessions,'fresh_generation_sessions':[s for s in sessions if s['processed']==69 and s['resumed']==0],'rows':rows}
    print(model_key, json.dumps({'records':len(rows),'statuses':counts,'score_eligible':models[model_key]['whole_output_score_eligible_count']}))
total=collections.Counter()
for model in models.values():total.update(model['status_counts'])
assert dict(total)=={'complete':260,'truncated':11,'invalid_evidence':5}
report={'scope':'Four-model saved static development record audit; no official correctness or routing claim','code_revision':CODE_REVISION,'validator_sha256':sha(pilot_path.read_bytes()),'total_records':276,'status_counts':dict(total),'models':models,'limitations':['Original records and frozen validation are unchanged.','Whole-output scores are unavailable for truncated or invalid-evidence records.','Saved parser outputs are inspected as metadata; official parsing/scoring and meaningful-token eligibility are separate.','Session aggregates retain resume overhead; fresh inference sessions are identified separately.','No routing threshold, calibration mapping or held-out evaluation is performed.']}
destination=root/'four-model-static-strata-audit-v1'
destination.mkdir(exist_ok=True)
output=destination/'record_audit.json'
output.write_text(json.dumps(report,indent=2)+'\n')
print('Total records:',report['total_records'],'Status counts:',report['status_counts'])
print('Audit report:',output)
from google.colab import files
files.download(str(output))
